# Crescimento populacional 2010 → 2022 por estado e por município

**Fonte:** IBGE — Censo Demográfico 2022: População e Domicílios, Primeiros Resultados (22/12/2023). Tabela com população 2010 recenseada, população 2010 compatibilizada para a base territorial de 2022 e população do Censo 2022.

Etapas:
1. Ler a tabela
2. Agregar a população por estado e ordenar pelo maior crescimento entre 2010 e 2022 → `populacao_estados.csv`
3. Trabalhar por município e ordenar pelo maior crescimento → `populacao_municipios.csv`

In [1]:
import numpy as np
import pandas as pd

## 1. Leitura da tabela

In [2]:
ARQUIVO = 'CD2022_Populacao_2010_Compatibilizada_20231222.xlsx'

# Linhas 0-1 têm título; o cabeçalho está na linha 2 (índice) e a coluna A está vazia
df = pd.read_excel(ARQUIVO, header=2, usecols='B:H', dtype={'COD. UF': str, 'COD. MUNIC': str})
df.head()

,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População Município 2010\n(Sinopse),População 2010 (Alterações de Limites até 2022)1,População Censo 2022
0,RO,11,00015,Alta Floresta D'Oeste,24392.0,24392.0,21494.0
1,RO,11,00023,Ariquemes,90353.0,90353.0,96833.0
2,RO,11,00031,Cabixi,6313.0,6313.0,5351.0
3,RO,11,00049,Cacoal,78574.0,78574.0,86887.0
4,RO,11,00056,Cerejeiras,17029.0,17029.0,15890.0


In [3]:
# Nomes de coluna mais curtos
df.columns = ['UF', 'COD_UF', 'COD_MUNIC', 'MUNICIPIO', 'POP_2010_SINOPSE', 'POP_2010', 'POP_2022']

# Remove as linhas de nota/fonte do rodapé (o texto fica na coluna UF, mas não têm código)
df = df.dropna(subset=['COD_UF']).copy()

colunas_numericas = ['POP_2010_SINOPSE', 'POP_2010', 'POP_2022']
df[colunas_numericas] = df[colunas_numericas].astype(np.int64)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   UF                5570 non-null   str  
 1   COD_UF            5570 non-null   str  
 2   COD_MUNIC         5570 non-null   str  
 3   MUNICIPIO         5570 non-null   str  
 4   POP_2010_SINOPSE  5570 non-null   int64
 5   POP_2010          5570 non-null   int64
 6   POP_2022          5570 non-null   int64
dtypes: int64(3), str(4)
memory usage: 304.7 KB


In [4]:
print('Municípios:', len(df))
print('Estados:', df['UF'].nunique())
df[colunas_numericas].sum()

Municípios: 5570
Estados: 27


POP_2010_SINOPSE    190755799
POP_2010            190755799
POP_2022            203080756
dtype: int64

> **Qual população de 2010 usar?** Conforme a nota do IBGE na própria planilha, a comparação 2010 × 2022 deve usar a coluna **População 2010 (Alterações de Limites até 2022)** — `POP_2010` aqui —, que reconstitui 2010 com os limites territoriais de 2022. Assim o crescimento não é inflado nem reduzido por mudanças de fronteira entre municípios. A coluna da Sinopse 2010 é mantida só como referência.

## 2. População agregada por estado

In [5]:
pop_estados = (
    df.groupby('UF')[['POP_2010_SINOPSE', 'POP_2010', 'POP_2022']]
    .sum()
    .reset_index()
)
pop_estados.head()

,UF,POP_2010_SINOPSE,POP_2010,POP_2022
0,AC,733559,733559,830018
1,AL,3120494,3120887,3127683
2,AM,3483985,3483985,3941613
3,AP,669526,669526,733759
4,BA,14016906,14017071,14141626


### Diferença 2022 − 2010 e ordenação pelo maior crescimento

In [6]:
pop_estados['CRESCIMENTO'] = pop_estados['POP_2022'] - pop_estados['POP_2010']
pop_estados['CRESCIMENTO_%'] = np.round(pop_estados['CRESCIMENTO'] / pop_estados['POP_2010'] * 100, 2)

pop_estados = pop_estados.sort_values('CRESCIMENTO', ascending=False).reset_index(drop=True)
pop_estados

,UF,POP_2010_SINOPSE,POP_2010,POP_2022,CRESCIMENTO,CRESCIMENTO_%
0,SP,41262199,41262199,44411238,3149039,7.63
1,SC,6248436,6248436,7610361,1361925,21.80
2,GO,6003788,6001789,7056495,1054706,17.57
3,PR,10444526,10444526,11444380,999854,9.57
4,MG,19597330,19597330,20539989,942659,4.81
5,MT,3035122,3035122,3658649,623527,20.54
6,PA,7581051,7581051,8120131,539080,7.11
7,AM,3483985,3483985,3941613,457628,13.14
8,CE,8452381,8451644,8794957,343313,4.06
9,ES,3514952,3514952,3833712,318760,9.07


In [7]:
pop_estados.to_csv(r'populacao_estados.csv', sep=';', decimal=',', encoding='utf-8-sig', index=False)

## 3. População por município

In [8]:
pop_municipios = df[['UF', 'COD_UF', 'COD_MUNIC', 'MUNICIPIO', 'POP_2010_SINOPSE', 'POP_2010', 'POP_2022']].copy()

# Código IBGE completo do município (UF + município, 7 dígitos)
pop_municipios['COD_IBGE'] = pop_municipios['COD_UF'] + pop_municipios['COD_MUNIC']

pop_municipios['CRESCIMENTO'] = pop_municipios['POP_2022'] - pop_municipios['POP_2010']
pop_municipios['CRESCIMENTO_%'] = np.round(pop_municipios['CRESCIMENTO'] / pop_municipios['POP_2010'] * 100, 2)

pop_municipios = pop_municipios.sort_values('CRESCIMENTO', ascending=False).reset_index(drop=True)
pop_municipios.head(20)

,UF,COD_UF,COD_MUNIC,MUNICIPIO,POP_2010_SINOPSE,POP_2010,POP_2022,COD_IBGE,CRESCIMENTO,CRESCIMENTO_%
0,AM,13,02603,Manaus,1802014,1802014,2063689,1302603,261675,14.52
1,DF,53,00108,Brasília,2570160,2572159,2817381,5300108,245222,9.53
2,SP,35,50308,São Paulo,11253503,11253503,11451999,3550308,198496,1.76
3,SP,35,52205,Sorocaba,586625,586816,723682,3552205,136866,23.32
4,GO,52,08707,Goiânia,1302001,1301912,1437366,5208707,135454,10.40
5,RR,14,00100,Boa Vista,284313,284313,413486,1400100,129173,45.43
6,SC,42,05407,Florianópolis,421240,421240,537211,4205407,115971,27.53
7,PA,15,05536,Parauapebas,153908,153908,267836,1505536,113928,74.02
8,MS,50,02704,Campo Grande,786797,786774,898100,5002704,111326,14.15
9,PB,25,07507,João Pessoa,723515,723515,833932,2507507,110417,15.26


Os municípios que mais **perderam** população ficam no fim da tabela:

In [9]:
pop_municipios.tail(10)

,UF,COD_UF,COD_MUNIC,MUNICIPIO,POP_2010_SINOPSE,POP_2010,POP_2022,COD_IBGE,CRESCIMENTO,CRESCIMENTO_%
5560,CE,23,04400,Fortaleza,2452185,2459712,2428708,2304400,-31004,-1.26
5561,RJ,33,01702,Duque de Caxias,855048,855088,808161,3301702,-46927,-5.49
5562,PE,26,11606,Recife,1537704,1537704,1488920,2611606,-48784,-3.17
5563,RN,24,08102,Natal,803739,803739,751300,2408102,-52439,-6.52
5564,MG,31,06200,Belo Horizonte,2375151,2375609,2315560,3106200,-60049,-2.53
5565,RS,43,14902,Porto Alegre,1409351,1409351,1332845,4314902,-76506,-5.43
5566,PA,15,01402,Belém,1393399,1393399,1303403,1501402,-89996,-6.46
5567,RJ,33,04904,São Gonçalo,999728,999728,896744,3304904,-102984,-10.30
5568,RJ,33,04557,Rio de Janeiro,6320446,6320446,6211223,3304557,-109223,-1.73
5569,BA,29,27408,Salvador,2675656,2675656,2417678,2927408,-257978,-9.64


In [10]:
pop_municipios.to_csv(r'populacao_municipios.csv', sep=';', decimal=',', encoding='utf-8-sig', index=False)

## 4. Resumo

In [11]:
print(f"Estados que cresceram: {(pop_estados['CRESCIMENTO'] > 0).sum()} de {len(pop_estados)}")
print(f"Municípios que cresceram: {(pop_municipios['CRESCIMENTO'] > 0).sum()} de {len(pop_municipios)}")
print(f"Municípios que perderam população: {(pop_municipios['CRESCIMENTO'] < 0).sum()}")
pop_municipios[['CRESCIMENTO', 'CRESCIMENTO_%']].describe()

Estados que cresceram: 27 de 27
Municípios que cresceram: 3172 de 5570
Municípios que perderam população: 2395


,CRESCIMENTO,CRESCIMENTO_%
count,5570.000000,5570.000000
mean,2212.739138,3.647591
std,11898.433572,14.530006
min,-257978.000000,-45.240000
25%,-349.750000,-4.680000
50%,125.000000,1.600000
75%,1280.750000,9.027500
max,261675.000000,188.510000
